# Notebook 3: Model Training

Step-by-step training of all four models with diagnostic plots:
- Random Forest — training and OOB error
- XGBoost — learning curve with early stopping
- LSTM — loss/accuracy per epoch
- DQN — reward, BLER, and throughput per episode

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

from src.channel.simulator import ChannelSimulator, ChannelConfig
from src.models.supervised import (
    CQIBaseline, LSTMWrapper, RFLinkAdapter, XGBLinkAdapter
)
from src.models.dqn_agent import DQNAgent, LinkAdaptationEnv

sns.set_theme(style='whitegrid', font_scale=1.1)
print('All imports OK')

## 1. Dataset Preparation

In [ ]:
cfg = ChannelConfig(channel_type='rayleigh', snr_mean_db=15.0, snr_std_db=6.0, seed=42)
sim = ChannelSimulator(cfg)
ds  = sim.generate_dataset(n_frames=60_000, window=8)

idx_tr, idx_val, idx_te = ds.train_val_test_split()
X_train, y_train = ds.features[idx_tr],  ds.labels_mcs[idx_tr]
X_val,   y_val   = ds.features[idx_val], ds.labels_mcs[idx_val]
X_test,  y_test  = ds.features[idx_te],  ds.labels_mcs[idx_te]
snr_test = ds.snr_trace[idx_te + 8]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Feature dim: {X_train.shape[1]}')

## 2. CQI Baseline (no training)

In [ ]:
cqi_model = CQIBaseline()
pred_cqi = cqi_model.predict(X_test)
acc_cqi = accuracy_score(y_test, pred_cqi)

print(f'CQI Baseline accuracy: {acc_cqi*100:.2f}%')
print('This is the baseline to beat. No training required — pure lookup table.')

## 3. Random Forest

In [ ]:
import time

print('Training Random Forest...')
t0 = time.time()

rf = RFLinkAdapter(n_estimators=300, max_depth=20)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, pred_rf)

print(f'Training time: {time.time()-t0:.1f}s')
print(f'Test accuracy: {acc_rf*100:.2f}%  (vs CQI baseline: {acc_cqi*100:.2f}%)')
print(f'Improvement: +{(acc_rf - acc_cqi)*100:.2f} percentage points')

# Feature importances
import pandas as pd
fi = pd.Series(rf.feature_importances_, index=ds.feature_names).sort_values(ascending=False)
print('\nTop 10 features:')
print(fi.head(10).to_string())

In [ ]:
# Plot feature importances
fig, ax = plt.subplots(figsize=(10, 6))
top_fi = fi.head(12)
ax.barh(top_fi.index[::-1], top_fi.values[::-1], color='#4C72B0', alpha=0.85)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest — Top Feature Importances')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/nb3_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. XGBoost

In [ ]:
print('Training XGBoost...')
t0 = time.time()

xgb_m = XGBLinkAdapter(n_estimators=400, max_depth=8, learning_rate=0.05)
xgb_m.fit(X_train, y_train, X_val, y_val)

pred_xgb = xgb_m.predict(X_test)
acc_xgb = accuracy_score(y_test, pred_xgb)

print(f'Training time: {time.time()-t0:.1f}s')
print(f'Test accuracy: {acc_xgb*100:.2f}%')

# Learning curve from eval history
evals = xgb_m.model.evals_result()
if evals:
    fig, ax = plt.subplots(figsize=(10, 4))
    for name, history in evals.items():
        ax.plot(history.get('mlogloss', []), label=name)
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('Multi-class Log Loss')
    ax.set_title('XGBoost Training / Validation Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/plots/nb3_xgb_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. LSTM Training

In [ ]:
print('Training LSTM (this takes a few minutes)...')
t0 = time.time()

lstm = LSTMWrapper(
    window=8, hidden_dim=128, num_layers=2,
    epochs=30, batch_size=512, device='cpu'
)
lstm.fit(X_train, y_train, X_val, y_val, verbose=True)

pred_lstm = lstm.predict(X_test)
acc_lstm = accuracy_score(y_test, pred_lstm)

print(f'\nTraining time: {time.time()-t0:.1f}s')
print(f'Test accuracy: {acc_lstm*100:.2f}%')

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(lstm.train_losses, color='#4C72B0', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].grid(alpha=0.3)

if lstm.val_accuracies:
    axes[1].plot(lstm.val_accuracies, color='#55A868', linewidth=1.5)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Accuracy')
    axes[1].set_title('LSTM Validation Accuracy')
    axes[1].grid(alpha=0.3)
    axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/plots/nb3_lstm_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. DQN Training

In [ ]:
print('Training DQN agent...')
t0 = time.time()

env_train = LinkAdaptationEnv(
    ds.features[idx_tr],
    ds.snr_trace[idx_tr + 8],
    channel_type='rayleigh'
)

agent = DQNAgent(
    state_dim=ds.features.shape[1],
    lr=1e-3, epsilon_decay=0.9995,
    batch_size=256, device='cpu'
)

history = agent.train(env_train, n_episodes=20, verbose=True)

# Greedy evaluation on test
pred_dqn = np.array([
    agent.select_action(X_test[i], greedy=True)
    for i in range(len(X_test))
])
acc_dqn = accuracy_score(y_test, pred_dqn)

print(f'\nTraining time: {time.time()-t0:.1f}s')
print(f'Test accuracy: {acc_dqn*100:.2f}%')

In [ ]:
# DQN training curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, label, color, scale in [
    (axes[0], 'episode_rewards', 'Episode Total Reward', '#4C72B0', 1.0),
    (axes[1], 'mean_bler',       'Mean BLER per Episode', '#C44E52', 1.0),
    (axes[2], 'mean_throughput', 'Mean Throughput (Mbps)', '#55A868', 1e-6),
]:
    data = np.array(history.get(key, [])) * scale
    ax.plot(data, color=color, linewidth=1.5, alpha=0.8, label='Per episode')
    if len(data) >= 5:
        ma = np.convolve(data, np.ones(5)/5, mode='valid')
        ax.plot(range(4, len(data)), ma, color='black', linewidth=2.5,
                label='5-ep MA', linestyle='--')
    ax.set_xlabel('Episode')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    # Epsilon overlay on reward plot
    if key == 'episode_rewards':
        ax2 = ax.twinx()
        eps_per_ep = [max(0.05, 1.0 * (0.9995 ** (i * len(X_train)))) for i in range(20)]
        ax2.plot(eps_per_ep, color='orange', linewidth=1, linestyle=':', label='ε')
        ax2.set_ylabel('Epsilon (ε)', color='orange')
        ax2.tick_params(axis='y', labelcolor='orange')

plt.suptitle('DQN Training Progress', fontsize=13)
plt.tight_layout()
plt.savefig('../results/plots/nb3_dqn_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Accuracy Summary

In [ ]:
summary = {
    'Model': ['CQI Baseline', 'Random Forest', 'XGBoost', 'LSTM', 'DQN'],
    'Accuracy (%)': [
        acc_cqi * 100, acc_rf * 100, acc_xgb * 100, acc_lstm * 100, acc_dqn * 100
    ]
}
import pandas as pd
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#937860', '#4C72B0', '#8172B2', '#55A868', '#C44E52']
bars = ax.bar(summary_df['Model'], summary_df['Accuracy (%)'],
              color=colors, alpha=0.85, edgecolor='white', width=0.6)
for bar, val in zip(bars, summary_df['Accuracy (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=9)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Model Accuracy Comparison')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/nb3_accuracy_summary.png', dpi=150, bbox_inches='tight')
plt.show()